# Diabetes Prediction using Logistic Regression
**Student:** KEERTHI H  
**Project:** Logistic Regression Project — Diabetes Prediction using Kaggle Data

This notebook follows the supplied project report. It predicts `Outcome` (0 = No diabetes, 1 = Diabetes) using the Pima Indians Diabetes Database (`diabetes.csv`).

**Workflow:** load data → inspect → replace invalid zeros → train-test split → median imputation → scaling → balanced Logistic Regression → evaluate → interpret coefficients.

Upload `diabetes.csv` when prompted in Google Colab, or place it beside this notebook in Jupyter.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay)

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 1. Load the dataset

In [ ]:
DATA_PATH = "diabetes.csv"
if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            DATA_PATH = next(iter(uploaded))
    except ImportError:
        DATA_PATH = input("Enter path to diabetes.csv: ").strip()

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())

## 2. Data inspection and audit

In [ ]:
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique()
}))
display(df.describe().T)
print("Duplicate rows:", df.duplicated().sum())
print("Columns:", df.columns.tolist())

## 3. Replace invalid zero values
For this dataset, zero is treated as an invalid measurement for Glucose, BloodPressure, SkinThickness, Insulin and BMI. These values are replaced with missing values and later median-imputed. The target Outcome and other features are not changed.

In [ ]:
target = "Outcome"
features = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
            "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]
required = features + [target]
missing_columns = sorted(set(required) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

data = df[required].copy()
invalid_zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in invalid_zero_cols:
    data[col] = data[col].replace(0, np.nan)

display(data.isna().sum().to_frame("missing_count"))

## 4. Feature-target separation and train-test split

In [ ]:
X = data[features]
y = data[target].astype(int)
display(y.value_counts().sort_index().rename(index={0: "No diabetes", 1: "Diabetes"}))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 5. Median imputation, scaling and Logistic Regression
The pipeline fits imputation and scaling on training data only. Balanced class weights help account for unequal class frequencies.

In [ ]:
model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=2000,
                                      random_state=RANDOM_STATE))
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print("Model training complete.")

## 6. Evaluation metrics

In [ ]:
metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}
display(pd.DataFrame({"Metric": metrics.keys(),
                      "Result": [round(v, 3) for v in metrics.values()]}))
print(classification_report(y_test, y_pred,
      target_names=["No diabetes", "Diabetes"], zero_division=0))

## 7. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=["No diabetes", "Diabetes"]).plot(
    cmap="Blues", values_format="d")
plt.title("Confusion Matrix — Logistic Regression")
plt.tight_layout()
plt.show()
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

## 8. ROC curve

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC Curve — Logistic Regression")
plt.tight_layout()
plt.show()

## 9. Interpret model coefficients

In [ ]:
classifier = model.named_steps["classifier"]
coef_df = pd.DataFrame({"Feature": features,
                        "Coefficient": classifier.coef_[0]}).sort_values(
                            "Coefficient", ascending=False)
display(coef_df)
plt.figure(figsize=(9, 5))
sns.barplot(data=coef_df, x="Coefficient", y="Feature")
plt.axvline(0, color="black", linewidth=1)
plt.title("Standardized Logistic Regression Coefficients")
plt.tight_layout()
plt.show()
print("Positive coefficients increase estimated log-odds of Outcome=1;")
print("negative coefficients decrease estimated log-odds, holding other inputs constant.")

## 10. Supplied report reference results
The uploaded report states these reference metrics. They are shown for comparison; your run may differ if the original report used a different split or model configuration.

| Metric | Reference result |
|---|---:|
| Accuracy | 0.734 |
| Precision | 0.603 |
| Recall | 0.704 |
| F1-score | 0.650 |
| ROC-AUC | 0.813 |

Reference confusion matrix: TN = 75, FP = 25, FN = 16, TP = 38.

## 11. Conclusion
This notebook implements median imputation, standardization and class-balanced Logistic Regression. Use the metrics and plots generated by your actual run when writing the final conclusion. Recall is important in a screening-style classification task because false negatives are positive cases that the model fails to identify.

The supplied report identifies Glucose and BMI as strong positive coefficients; verify the coefficient ranking from this notebook before repeating that finding.

**Limitation:** This small historical dataset represents a specific population. This project is for learning classification concepts and must not be used for medical diagnosis.